# Titanic

**Objetivo:** Classificar os passageiros de um cruzeiro quanto a probabilidade de sobrevivência, para embasar precificação de seguros.

## 1. Imports

In [ ]:
from sklearn.datasets import fetch_openml
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
import random

seed = 42
random.seed(seed)

In [ ]:
titanic = fetch_openml("titanic", version=1, as_frame=True).frame

titanic['survived'] = pd.to_numeric(titanic['survived'])
titanic['pclass'] = pd.to_numeric(titanic['pclass'])

In [ ]:
titanic = titanic[['pclass', 'survived', "age", "sibsp", "parch", 'fare']]

2. Análise Exploratória dos Dados

In [ ]:
# Formato dos dados
titanic.shape

(1309, 14)

In [ ]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   int64  
 1   survived  1309 non-null   int64  
 2   age       1046 non-null   float64
 3   sibsp     1309 non-null   int64  
 4   parch     1309 non-null   int64  
 5   fare      1308 non-null   float64
dtypes: float64(2), int64(4)
memory usage: 61.5 KB


In [ ]:
titanic.describe()

,pclass,survived,age,sibsp,parch,fare
count,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000
mean,2.294882,0.381971,29.881135,0.498854,0.385027,33.295479
std,0.837836,0.486055,14.413500,1.041658,0.865560,51.758668
min,1.000000,0.000000,0.166700,0.000000,0.000000,0.000000
25%,2.000000,0.000000,21.000000,0.000000,0.000000,7.895800
50%,3.000000,0.000000,28.000000,0.000000,0.000000,14.454200
75%,3.000000,1.000000,39.000000,1.000000,0.000000,31.275000
max,3.000000,1.000000,80.000000,8.000000,9.000000,512.329200


In [ ]:
titanic.isnull().sum()

,0
pclass,0
survived,0
age,263
sibsp,0
parch,0
fare,1


In [ ]:
titanic['survived'].value_counts(normalize=True)

,proportion
survived,
0,0.618029
1,0.381971


## 3. Pré - Processamento dos *Dados*

In [ ]:
titanic.dropna(subset=["fare"], inplace=True)

In [ ]:
X = titanic.drop('survived', axis=1)
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
def feature_engineering(X):
  X = X.copy()
  X['age']     = pd.cut(X['age'],
                        bins=[0, 12, 60, 100],
                        labels=[1,2,3]).astype(int)
  return X

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Inclui a etapa de imputação dentro do pipeline, aplicando apenas às colunas selecionadas
preproc = ColumnTransformer(transformers=[
    ('imputer', SimpleImputer(strategy='median'), ["age"]),
    # Qualquer pre-processamento
], remainder='passthrough' # Deixa as colunas não listadas inalteradas
, verbose_feature_names_out=False  # Evita prefixos nas colunas transformadas
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


# Pipeline base (parte do pré-processamento, reutilizada em ambos os modelos)
def make_pipeline(classifier):
    return Pipeline([
        ('preproc',       preproc),
        ('feature_eng',   FunctionTransformer(
            feature_engineering, feature_names_out="one-to-one")),
        ('clf',           classifier),
    ])
from sklearn import set_config

set_config(transform_output="pandas")

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV

dt_pipe_0 = make_pipeline(DecisionTreeClassifier(random_state=42))

dt_pipe_0.fit(X_train, y_train)

In [ ]:
dt_pipe_0 = make_pipeline(DecisionTreeClassifier(random_state=42))

dt_pipe_0.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preproc',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputer',
                                                  SimpleImputer(strategy='median'),
                                                  ['age'])],
                                   verbose_feature_names_out=False)),
                ('feature_eng',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<function feature_engineering at 0x7f6cc89568e0>)),
                ('clf', DecisionTreeClassifier(random_state=42))])

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, dt_pipe_0.predict(X_test))

0.6564885496183206

In [ ]:
dt_pipe_0.score(X_test, y_test)

0.6564885496183206

In [ ]:
dt_pipe_1 = make_pipeline(DecisionTreeClassifier(random_state=42))

dt_grid = {
    'clf__max_depth':         [3, 4, 5, 6, 7, 8, 10, None],
    'clf__min_samples_leaf':  [1, 5, 10, 20, 30],
    'clf__min_samples_split': [2, 5, 10],
    'clf__criterion':         ['gini', 'entropy'],
}

dt_search = GridSearchCV(
    dt_pipe_1, dt_grid,
    cv=5, scoring='accuracy'
)
dt_search.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preproc',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('imputer',
                                                                         SimpleImputer(strategy='median'),
                                                                         ['age'])],
                                                          verbose_feature_names_out=False)),
                                       ('feature_eng',
                                        FunctionTransformer(feature_names_out='one-to-one',
                                                            func=<function feature_engineering at 0x7f6cc89568e0>)),
                                       ('clf',
                                        DecisionTreeClassifier(random_state=42))]),
             param_grid={'clf__criterion': ['gini', 'entropy'],
                         'clf__max_depth': [3, 4, 5, 6, 7, 8, 10, None],
                         'clf__min_samples_leaf': [1, 5, 10, 20, 30],
                         'clf__min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [ ]:
print(f"Total de combinações testadas: {len(dt_search.cv_results_['params'])}")
print(f"Melhor CV score: {dt_search.best_score_:.4f}")
print(f"Melhores parâmetros:")
for k, v in dt_search.best_params_.items():
    print(f"  {k}: {v}")

Total de combinações testadas: 240
Melhor CV score: 0.7180
Melhores parâmetros:
  clf__criterion: entropy
  clf__max_depth: 7
  clf__min_samples_leaf: 5
  clf__min_samples_split: 2


In [ ]:
params = {'criterion': 'entropy',
 'max_depth': 7,
 'min_samples_leaf': 5,
 'min_samples_split': 2}

In [ ]:
dt_pipe_2 = make_pipeline(DecisionTreeClassifier(**params, random_state=42))

dt_pipe_2.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preproc',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputer',
                                                  SimpleImputer(strategy='median'),
                                                  ['age'])],
                                   verbose_feature_names_out=False)),
                ('feature_eng',
                 FunctionTransformer(feature_names_out='one-to-one',
                                     func=<function feature_engineering at 0x7f6cc89568e0>)),
                ('clf',
                 DecisionTreeClassifier(criterion='entropy', max_depth=7,
                                        min_samples_leaf=5, random_state=42))])

In [ ]:
pred = dt_pipe_2.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, pred)

0.7213740458015268

## Salvar Modelo

In [ ]:
# Savar model
import joblib

joblib.dump(dt_pipe_2, "titanic_model.joblib", compress=3)

['titanic_model.joblib']

# Carregar o Modelo Treinado

In [ ]:
index_inferencia = [54, 286, 361, 1173, 239, 123, 933, 873, 51, 969, 1033, 862, 31, 1301,
       367]

In [ ]:
dados_inferencia = X_test.loc[[1189, 123, 816, 858, 363, 802, 439, 538, 371, 844, 1285, 788, 430, 409,
       210]]

In [ ]:
loaded_model = joblib.load("titanic_model.joblib")

# Predict
# X_test é passado bruto — o pipeline aplica todas as transformações automaticamente
inferencia_pred = loaded_model.predict(dados_inferencia)
print(f"{len(inferencia_pred)} predições geradas para o conjunto de teste.")

15 predições geradas para o conjunto de teste.
